# Day 5 multi-token prediction on Kaggle

Attach the private `octlm-code` dataset and the completed `octlm-day4` and `octlm-day5-spread` notebook outputs. Run with one T4 after the seed-spread notebook finishes.


In [ ]:
import subprocess
import sys
from pathlib import Path

import torch

INPUT = Path("/kaggle/input")
CODE = next(path.parent.parent for path in INPUT.rglob("day5.py") if path.parent.name == "octlm")
TRAIN = next(path for path in INPUT.rglob("train.bin") if path.parent.name == "data")
DATA = TRAIN.parent
TOKENIZER = DATA.parent / "bpe.json"
SPREAD = next(INPUT.rglob("seed-spread.jsonl"))
OUT = Path("/kaggle/working/day5/mtp.jsonl")
OUT.parent.mkdir(parents=True, exist_ok=True)
if not torch.cuda.is_available() or torch.cuda.get_device_capability(0) != (7, 5):
    raise RuntimeError("Select a Kaggle T4 GPU")
print("PyTorch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0), flush=True)
print("Control rows:", len(SPREAD.read_text().splitlines()), flush=True)

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "octlm.day5",
        "mtp",
        "--config",
        str(CODE / "configs/day5.toml"),
        "--data",
        str(DATA),
        "--tokenizer",
        str(TOKENIZER),
        "--spread-in",
        str(SPREAD),
        "--out",
        str(OUT),
        "--device",
        "cuda:0",
    ],
    cwd=CODE,
    check=True,
)
print(OUT.read_text(), flush=True)